# HBC：加载模型或训练


默认运行：有完整峰值权重则直接加载峰值模型，否则加载现有末轮模型，不会为了补存峰值自动重训。
只有模型缺失时才训练；新训练会在每次 ARI 创新高时保存对应权重，并另外保留末轮权重。

旧实验没有保存峰值权重，不能从末轮模型逆推出。需要补存时，将下方 `TRAIN_FOR_BEST = True`；程序按原参数和预算重跑缺少峰值权重的 seed，保留原末轮模型与历史归档。完成后再次运行会跳过训练。

运行结果将明确显示加载的是峰值还是末轮模型、实际轮次、各 seed 指标及与历史结果的差异。补存后的结果是新训练结果，不能保证与原峰值逐位相同。


In [1]:
from pathlib import Path
import sys, importlib.util
ROOT = next((q for p in [Path.cwd(), *Path.cwd().parents]
             for q in [p, p / "sidmgf_final"] if (q / "config/paper.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请在 sidmgf_final 或其 tutorials 目录中打开 Notebook")
spec = importlib.util.spec_from_file_location("spasrd_notebook_output", ROOT / "src/notebook_output.py")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
RESULTS = module.NotebookResults(ROOT)
execute = RESULTS.execute
print("项目目录:", ROOT)
print("运行日志、指标表和图件预览将直接显示在本 Notebook 中。")


项目目录: c:\Users\吴浪\Desktop\ST\sidmgf_final
运行日志、指标表和图件预览将直接显示在本 Notebook 中。


In [2]:
DATASET = "HBC"
PROFILE = "refined"  # HBC 可另选 refined；其余数据保持 paper
DEVICE = "auto"
MODEL_CHECKPOINT = "auto"  # 优先加载完整峰值权重；没有时加载原末轮权重
TRAIN_FOR_BEST = True  # 改为 True：仅对缺少完整峰值权重的 seed 重跑补存
args = ["run", "--dataset", DATASET, "--profile", PROFILE,
        "--device", DEVICE, "--model-checkpoint", MODEL_CHECKPOINT]
if TRAIN_FOR_BEST:
    args.append("--recover-best")
execute(*args)


开始执行：run --dataset HBC --profile refined --device auto --model-checkpoint auto --recover-best
[1/5]
C:\Users\吴浪\Desktop\ST\sidmgf_final\src\workflow.py:62: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  return torch.sparse_coo_tensor(np.vstack((coo.row,coo.col)), coo.data,
补存峰值权重：按原配置重新训练；保留原末轮权重和历史归档。
保存峰值模型：epoch=1, ARI=0.336724, seed_0_best.pt
保存峰值模型：epoch=2, ARI=0.370856, seed_0_best.pt
保存峰值模型：epoch=3, ARI=0.433708, seed_0_best.pt
保存峰值模型：epoch=6, ARI=0.445958, seed_0_best.pt
保存峰值模型：epoch=7, ARI=0.460664, seed_0_best.pt
保存峰值模型：epoch=12, ARI=0.468116, seed_0_best.pt
保存峰值模型：epoch=13, ARI=0.469677, see

**本次完成 5 个实验：加载已有模型 0 个，本次训练 5 个。**

**本次预测：逐 seed 指标、实际加载轮次与历史对照**

数据,样本,方案,seed,执行方式,加载权重类型,完整峰值权重,权重轮次,本次预测ARI,本次预测NMI,历史对照类型,历史对照ARI,与历史对照ARI差值,表示最大绝对差,聚类一致性ARI,与新存峰值ARI差值,归档峰值ARI（参考）
HBC,human_breast_cancer,refined,0,完成重跑，补存峰值权重,峰值模型,已保存,349,0.698231,0.681847,归档峰值,0.698231,0.000000,0.000000,1.000000,0.000000,0.698231
HBC,human_breast_cancer,refined,1,完成重跑，补存峰值权重,峰值模型,已保存,196,0.686405,0.678750,归档峰值,0.686405,0.000000,0.000000,1.000000,0.000000,0.686405
HBC,human_breast_cancer,refined,2,完成重跑，补存峰值权重,峰值模型,已保存,188,0.695703,0.692359,归档峰值,0.695703,0.000000,0.000000,1.000000,0.000000,0.695703
HBC,human_breast_cancer,refined,3,完成重跑，补存峰值权重,峰值模型,已保存,187,0.685123,0.684665,归档峰值,0.685123,0.000000,0.000000,1.000000,0.000000,0.685123
HBC,human_breast_cancer,refined,42,完成重跑，补存峰值权重,峰值模型,已保存,350,0.713797,0.709136,归档峰值,0.713797,0.000000,0.000000,1.000000,0.000000,0.713797


**本次预测：逐样本均值与标准差**

数据,样本,方案,加载权重类型,seed数,ARI均值,ARI标准差,NMI均值,NMI标准差
HBC,human_breast_cancer,refined,峰值模型,5,0.695852,0.011533,0.689351,0.012157


本次使用的模型以“加载权重类型”和“权重轮次”为准；不同类型分开汇总。重跑获得的峰值可能与历史归档不同。“与新存峰值ARI差值”用于核对峰值权重重新加载后能否恢复新训练的峰值。聚类一致性 ARI=1 表示分区一致，表示最大绝对差=0 表示表示数值完全一致。

In [3]:
execute("summary", "--dataset", DATASET, "--profile", PROFILE)


开始执行：summary --dataset HBC --profile refined


**历史归档：逐样本汇总**

数据,样本,方案,seed数,归档峰值ARI均值,峰值ARI标准差,归档末轮ARI均值,末轮ARI标准差
HBC,human_breast_cancer,refined,5,0.695852,0.011533,0.640550,0.025957


**历史归档：数据集汇总**

数据,方案,结果类型,统计单位,数量,ARI均值,ARI中位数,ARI标准差
HBC,refined,归档峰值,seed,5,0.695852,0.695703,0.011533
HBC,refined,归档末轮,seed,5,0.640550,0.637607,0.025957


峰值是各 seed 训练过程中最高 ARI，**不是当前末轮权重重新预测的结果**。DLPFC 先对每片的 seed 取均值，再汇总切片；标准差均为样本标准差。